# BNBUSDT — Progressive Reinvestment Backtest

Comparing 3 candidate parameter sets from grid search (70% sell at ATH).

In [ ]:
import pandas as pd
import requests, time, numpy as np, matplotlib.pyplot as plt
from datetime import datetime, timezone

plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 9

In [ ]:
def fetch_monthly_klines(symbol='BNBUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data: break
        all_klines.extend(data)
        if len(data) < limit: break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines('BNBUSDT')
cols = ['timestamp','open','high','low','close','volume','close_time','quote_vol','trades','taker_buy_base','taker_buy_quote','ignore']
df = pd.DataFrame(klines, columns=cols)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open','high','low','close','volume']: df[c] = df[c].astype(float)
df = df[['date','open','high','low','close','volume']].copy().sort_values('date').reset_index(drop=True)
print(f'{df["date"].min().strftime("%Y-%m")} → {df["date"].max().strftime("%Y-%m")} ({len(df)} months)')

In [ ]:
def run_backtest(start_pct, increment, cap, label=""):
    coin = 0.0; usdt = 0.0; invested = 0.0; highest = 0.0; reinvest = start_pct
    records = []
    for _, row in df.iterrows():
        close = row['close']
        if close < row['open']:
            extra = usdt * (reinvest / 100.0) if usdt > 0 else 0.0
            coin += (10.0 + extra) / close
            invested += 10.0
            usdt -= extra
        prev = highest
        if close > highest: highest = close
        if coin > 0.000001 and close > prev:
            sell = coin * 0.7
            usdt += sell * close
            coin -= sell
            reinvest = start_pct
        elif reinvest < cap:
            reinvest = min(cap, reinvest + increment)
        records.append({'date': row['date'], 'close': close, 'coin': coin, 'usdt': usdt,
                        'value': coin * close + usdt, 'invested': invested})
    r = pd.DataFrame(records)
    fv = r['value'].iloc[-1]
    ret = (fv / invested - 1) * 100
    eq = r['value']
    dd = ((eq.cummax() - eq) / eq.cummax()).max() * 100
    m = eq.pct_change().dropna()
    m = m[m.index >= 12]
    sharpe = (m.mean() / m.std()) * np.sqrt(12) if m.std() > 0 else 0
    pos = m[m > 0].sum(); neg = m[m < 0].abs().sum()
    pf = pos / neg if neg > 0 else float('inf')
    ann = (1 + ret/100) ** (12/len(df)) - 1
    calmar = ann / (dd/100) if dd > 0 else 0
    return r, {'label': label, 'return': ret, 'dd': dd, 'sharpe': sharpe, 'calmar': calmar, 'pf': pf,
                'final': fv, 'invested': invested, 'coin': coin, 'usdt': usdt}

params = [
    (1, 1, 30, '1%+1%->30% (best Sharpe)'),
    (1, 1, 40, '1%+1%->40% (best Calmar)'),
    (3, 1, 50, '3%+1%->50% (low DD high R)'),
    (1, 2, 50, '1%+2%->50% (BTC optimum)'),
]
results = {}
for st, inc, cp, lbl in params:
    r, m = run_backtest(st, inc, cp, lbl)
    results[lbl] = (r, m)
    print(f"{lbl:>30s}  R={m['return']:6.1f}%  DD={m['dd']:5.1f}%  Sharpe={m['sharpe']:.2f}  Calmar={m['calmar']:.2f}  PF={m['pf']:.2f}  Final=${m['final']:.0f}")

In [ ]:
# Monthly detail for best Calmar params

r, m = results['1%+1%->40% (best Calmar)']
detail = r.copy()
detail['open'] = df['open']
detail['red'] = detail['close'] < detail['open']
detail['action'] = ''
for i in range(len(detail)):
    prev_close = detail['close'].iloc[i-1] if i > 0 else 0
    if i > 0 and detail['coin'].iloc[i] > detail['coin'].iloc[i-1]:
        diff = detail['coin'].iloc[i] - detail['coin'].iloc[i-1]
        detail.at[i, 'action'] = f'buy {diff:.6f}'
    elif i > 0 and detail['coin'].iloc[i] < detail['coin'].iloc[i-1]:
        diff = detail['coin'].iloc[i-1] - detail['coin'].iloc[i]
        detail.at[i, 'action'] = f'sell {diff:.6f}'

print(f"{'Date':>10s} {'Close':>8s} {'Red':>5s} {'Coin':>10s} {'USDT':>10s} {'Value':>10s} {'Invested':>9s} {'Action':>20s}")
print("-"*87)
for _, row in detail.iterrows():
    print(f"{row['date'].strftime('%Y-%m'):>10s} {row['close']:>8.1f} {str(row['red']):>5s} {row['coin']:>10.6f} {row['usdt']:>10.2f} {row['value']:>10.2f} {row['invested']:>9.0f} {row['action']:>20s}")

In [ ]:
# Equity curves comparison

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
dates = results[params[0][3]][0]['date']

for idx, (*_, lbl) in enumerate(params):
    r, m = results[lbl]
    ax = axes[0]
    ax.plot(dates, r['value'], label=f"{lbl}  (R={m['return']:.1f}%)", color=colors[idx], linewidth=1.5)
axes[0].set_title('Portfolio Value ($10/month base)')
axes[0].set_ylabel('Value (USDT)')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

for idx, (*_, lbl) in enumerate(params):
    r, m = results[lbl]
    ax = axes[1]
    eq = r['value']
    dd = (eq.cummax() - eq) / eq.cummax() * 100
    ax.fill_between(dates, 0, dd, color=colors[idx], alpha=0.3, label=f"{lbl}")
    ax.plot(dates, dd, color=colors[idx], linewidth=0.8)
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown (%)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

ax = axes[2]
ax.plot(dates, df['close'], color='gray', linewidth=1.5, alpha=0.7)
ax.set_title('BNBUSDT Monthly Close Price')
ax.set_ylabel('Price (USDT)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('dca-bt-buy-bot-bnb-comparison.png', dpi=150, bbox_inches='tight')
print('Saved dca-bt-buy-bot-bnb-comparison.png')
plt.show()

In [ ]:
# Summary table

rows = []
for *_, lbl in params:
    r, m = results[lbl]
    rows.append(m)
summary = pd.DataFrame(rows)
summary = summary[['label', 'return', 'dd', 'sharpe', 'calmar', 'pf', 'final', 'invested', 'coin', 'usdt']]
summary.columns = ['Strategy', 'Return%', 'Max DD%', 'Sharpe', 'Calmar', 'Profit Factor', 'Final Value', 'Total Invested', 'Coin Held', 'USDT Reserve']
print(summary.to_string(index=False))

In [ ]:
# Additional: compare with HODL and monthly DCA

first_close = df['close'].iloc[0]
last_close = df['close'].iloc[-1]
hodl_coin = sum(10.0 / df.loc[df['close'] < df['open'], 'close'])  # buy $10 each red month
hodl_final = hodl_coin * last_close
hodl_ret = (hodl_final / (len(df[df['close'] < df['open']]) * 10) - 1) * 100

dca_coin = sum(10.0 / df['close'])  # buy $10 every month regardless
dca_final = dca_coin * last_close
dca_ret = (dca_final / (len(df) * 10) - 1) * 100

r, m = results['1%+1%->40% (best Calmar)']
print(f"{'Strategy':>30s}  {'Return':>8s}  {'Final':>10s}")
print("-"*52)
print(f"{'Red-only HODL':>30s}  {hodl_ret:>7.1f}%  ${hodl_final:>8.0f}")
print(f"{'Monthly DCA (all months)':>30s}  {dca_ret:>7.1f}%  ${dca_final:>8.0f}")
print(f"{'1%+1%->40% (best Calmar)':>30s}  {m['return']:>7.1f}%  ${m['final']:>8.0f}")
print(f"{'1%+2%->50% (BTC opt)':>30s}  {results['1%+2%->50% (BTC optimum)'][1]['return']:>7.1f}%  ${results['1%+2%->50% (BTC optimum)'][1]['final']:>8.0f}")
print(f"{'3%+1%->50% (low DD high R)':>30s}  {results['3%+1%->50% (low DD high R)'][1]['return']:>7.1f}%  ${results['3%+1%->50% (low DD high R)'][1]['final']:>8.0f}")